# Data Preprocessing

In [1]:
import findspark
findspark.init()
import pyspark #only run after findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Data Preprocessing').getOrCreate()
spark

from pyspark.ml.feature import VectorAssembler
from pyspark.mllib.feature import StandardScaler, PCA
from pyspark.mllib.stat import Statistics

import numpy as np
import pandas as pd

#### Read the dataset

In [2]:
# Read the dataset
df = spark.read.csv('DailyDemandForecastingOrdersEditSJ.csv', header = True, inferSchema = True)
df.show()
df.printSchema()

+------------------------------------------------------------------+----------------------------------+----------------+------------+------------+------------+------------+--------------------+-----------------------------------------+------------------+------------------+------------------+---------------------+
|Week of the month (first week, second, third, fourth or fifth week|Day of the week (Monday to Friday)|Non-urgent order|Urgent order|Order type A|Order type B|Order type C|Fiscal sector orders|Orders from the traffic controller sector|Banking orders (1)|Banking orders (2)|Banking orders (3)|Target (Total orders)|
+------------------------------------------------------------------+----------------------------------+----------------+------------+------------+------------+------------+--------------------+-----------------------------------------+------------------+------------------+------------------+---------------------+
|                                                      

#### Shorten the features name

In [3]:
from functools import reduce

oldColumns = df.schema.names
newColumns = ['WkOfMth','DayOfWk','NonUrgentOrd','UrgentOrd','OrdTpA','OrdTpB','OrdTpC','FiscalSecOrd','OrdFrmTrafCtrl','BankOrd1','BankOrd2','BankOrd3','Target']

df1 = reduce(lambda df1, idx: df1.withColumnRenamed(oldColumns[idx], newColumns[idx]),range(len(oldColumns)), df)
df1.show()
df1.printSchema()

+-------+-------+------------+---------+------+-------+-------+------------+--------------+--------+--------+--------+-------+
|WkOfMth|DayOfWk|NonUrgentOrd|UrgentOrd|OrdTpA| OrdTpB| OrdTpC|FiscalSecOrd|OrdFrmTrafCtrl|BankOrd1|BankOrd2|BankOrd3| Target|
+-------+-------+------------+---------+------+-------+-------+------------+--------------+--------+--------+--------+-------+
|      1|      4|     316.307|   223.27|61.543|175.586|302.448|         0.0|         65556|   44914|  188411|   14793|539.577|
|      1|      5|     128.633|   96.042|38.058| 56.037| 130.58|         0.0|         40419|   21399|   89461|    7679|224.675|
|      1|      6|      43.651|   84.375|21.826| 25.125| 82.461|       1.386|         11992|    3452|   21305|   14947|129.412|
|      2|      2|     171.297|  127.667|41.542|113.294|162.284|      18.156|         49971|   33703|   69054|   18423| 317.12|
|      2|      3|      90.532|  113.526|37.679| 56.618| 116.22|       6.459|         48534|   19646|   16411|  

#### Checking for missing values

In [4]:
# Check if there is any NULL values. If yes, remove them
from pyspark.sql.functions import col, count, isnan, when
df1.select([count(when(col(c).isNull(), c)).alias(c) for c in df1.columns]).show()

+-------+-------+------------+---------+------+------+------+------------+--------------+--------+--------+--------+------+
|WkOfMth|DayOfWk|NonUrgentOrd|UrgentOrd|OrdTpA|OrdTpB|OrdTpC|FiscalSecOrd|OrdFrmTrafCtrl|BankOrd1|BankOrd2|BankOrd3|Target|
+-------+-------+------------+---------+------+------+------+------------+--------------+--------+--------+--------+------+
|      1|      1|           1|        2|     1|     1|     1|           2|             1|       1|       1|       1|     1|
+-------+-------+------------+---------+------+------+------+------------+--------------+--------+--------+--------+------+



#### Remove null value records

In [5]:
df1 = df1.na.drop(how = 'all')

# Check if there is any NULL values. If yes, remove them
from pyspark.sql.functions import col, count, isnan, when
df1.select([count(when(col(c).isNull(), c)).alias(c) for c in df1.columns]).show()

+-------+-------+------------+---------+------+------+------+------------+--------------+--------+--------+--------+------+
|WkOfMth|DayOfWk|NonUrgentOrd|UrgentOrd|OrdTpA|OrdTpB|OrdTpC|FiscalSecOrd|OrdFrmTrafCtrl|BankOrd1|BankOrd2|BankOrd3|Target|
+-------+-------+------------+---------+------+------+------+------------+--------------+--------+--------+--------+------+
|      0|      0|           0|        1|     0|     0|     0|           1|             0|       0|       0|       0|     0|
+-------+-------+------------+---------+------+------+------+------------+--------------+--------+--------+--------+------+



#### Treating Missing Values
Imputer:
Imputation estimator for completing missing values, using the mean, median or mode of the columns in which the missing 
values are located. The input columns should be of numeric type. Currently Imputer does not support categorical features 
and possibly creates incorrect values for a categorical feature.

In [6]:
# Import Imputer class from PySpark

from pyspark.ml.feature import Imputer

# Create an instance of imputer for all missing values (numeric fields)
myImputer = Imputer(
                inputCols = ['UrgentOrd', 'FiscalSecOrd'],
                outputCols = ['UrgentOrd', 'FiscalSecOrd']
).setStrategy('mean')

df1 = myImputer.fit(df1).transform(df1)

df1.show()

+-------+-------+------------+------------------+------+-------+-------+------------+--------------+--------+--------+--------+-------+
|WkOfMth|DayOfWk|NonUrgentOrd|         UrgentOrd|OrdTpA| OrdTpB| OrdTpC|FiscalSecOrd|OrdFrmTrafCtrl|BankOrd1|BankOrd2|BankOrd3| Target|
+-------+-------+------------+------------------+------+-------+-------+------------+--------------+--------+--------+--------+-------+
|      1|      4|     316.307|            223.27|61.543|175.586|302.448|         0.0|         65556|   44914|  188411|   14793|539.577|
|      1|      5|     128.633|            96.042|38.058| 56.037| 130.58|         0.0|         40419|   21399|   89461|    7679|224.675|
|      1|      6|      43.651|            84.375|21.826| 25.125| 82.461|       1.386|         11992|    3452|   21305|   14947|129.412|
|      2|      2|     171.297|           127.667|41.542|113.294|162.284|      18.156|         49971|   33703|   69054|   18423| 317.12|
|      2|      3|      90.532|           113.526

In [7]:
# Check if there is any NULL values. If yes, remove them
from pyspark.sql.functions import col, count, isnan, when
df1.select([count(when(col(c).isNull(), c)).alias(c) for c in df1.columns]).show()

+-------+-------+------------+---------+------+------+------+------------+--------------+--------+--------+--------+------+
|WkOfMth|DayOfWk|NonUrgentOrd|UrgentOrd|OrdTpA|OrdTpB|OrdTpC|FiscalSecOrd|OrdFrmTrafCtrl|BankOrd1|BankOrd2|BankOrd3|Target|
+-------+-------+------------+---------+------+------+------+------------+--------------+--------+--------+--------+------+
|      0|      0|           0|        0|     0|     0|     0|           0|             0|       0|       0|       0|     0|
+-------+-------+------------+---------+------+------+------+------------+--------------+--------+--------+--------+------+



In [8]:
df1.corr('UrgentOrd','FiscalSecOrd')

-0.004690352146447749

In [9]:
features = df1.drop('Target')

In [10]:
# We need to convert dataframe into an RDD to check for correlation
colName = features.columns
featuresRDD = features.rdd
featuresRDD.collect()

[Row(WkOfMth=1, DayOfWk=4, NonUrgentOrd=316.307, UrgentOrd=223.27, OrdTpA=61.543, OrdTpB=175.586, OrdTpC=302.448, FiscalSecOrd=0.0, OrdFrmTrafCtrl=65556, BankOrd1=44914, BankOrd2=188411, BankOrd3=14793),
 Row(WkOfMth=1, DayOfWk=5, NonUrgentOrd=128.633, UrgentOrd=96.042, OrdTpA=38.058, OrdTpB=56.037, OrdTpC=130.58, FiscalSecOrd=0.0, OrdFrmTrafCtrl=40419, BankOrd1=21399, BankOrd2=89461, BankOrd3=7679),
 Row(WkOfMth=1, DayOfWk=6, NonUrgentOrd=43.651, UrgentOrd=84.375, OrdTpA=21.826, OrdTpB=25.125, OrdTpC=82.461, FiscalSecOrd=1.386, OrdFrmTrafCtrl=11992, BankOrd1=3452, BankOrd2=21305, BankOrd3=14947),
 Row(WkOfMth=2, DayOfWk=2, NonUrgentOrd=171.297, UrgentOrd=127.667, OrdTpA=41.542, OrdTpB=113.294, OrdTpC=162.284, FiscalSecOrd=18.156, OrdFrmTrafCtrl=49971, BankOrd1=33703, BankOrd2=69054, BankOrd3=18423),
 Row(WkOfMth=2, DayOfWk=3, NonUrgentOrd=90.532, UrgentOrd=113.526, OrdTpA=37.679, OrdTpB=56.618, OrdTpC=116.22, FiscalSecOrd=6.459, OrdFrmTrafCtrl=48534, BankOrd1=19646, BankOrd2=16411, Ba

In [11]:
featuresRDD = features.rdd.map(lambda row: row[0:])
featuresRDD.collect()

[(1,
  4,
  316.307,
  223.27,
  61.543,
  175.586,
  302.448,
  0.0,
  65556,
  44914,
  188411,
  14793),
 (1,
  5,
  128.633,
  96.042,
  38.058,
  56.037,
  130.58,
  0.0,
  40419,
  21399,
  89461,
  7679),
 (1,
  6,
  43.651,
  84.375,
  21.826,
  25.125,
  82.461,
  1.386,
  11992,
  3452,
  21305,
  14947),
 (2,
  2,
  171.297,
  127.667,
  41.542,
  113.294,
  162.284,
  18.156,
  49971,
  33703,
  69054,
  18423),
 (2,
  3,
  90.532,
  113.526,
  37.679,
  56.618,
  116.22,
  6.459,
  48534,
  19646,
  16411,
  20257),
 (2,
  4,
  110.925,
  96.36,
  30.792,
  50.704,
  125.868,
  79.0,
  52042,
  8773,
  47522,
  24966),
 (2,
  5,
  144.124,
  118.919,
  43.304,
  66.371,
  153.368,
  0.0,
  46573,
  33597,
  48269,
  20973),
 (2,
  6,
  119.379,
  113.87,
  38.584,
  85.961,
  124.413,
  15.709,
  35033,
  26278,
  56665,
  18502),
 (3,
  2,
  218.856,
  124.381,
  33.973,
  148.274,
  162.044,
  1.054,
  66612,
  19461,
  103376,
  10458),
 (3,
  3,
  146.518,
  101.045,
 

#### Statistics

In [12]:
summary = Statistics.colStats(featuresRDD)
print(summary.mean())
print(summary.variance())
print(summary.numNonzeros())
print(summary.normL1())

[3.01666667e+00 4.03333333e+00 1.72554933e+02 1.18539746e+02
 5.21122167e+01 1.09229850e+02 1.39531250e+02 7.77308983e+01
 4.45043500e+04 4.66408333e+04 7.94014833e+04 2.31146333e+04]
[1.64378531e+00 1.96497175e+00 4.83105456e+03 7.29544930e+02
 3.54565533e+02 2.57468848e+03 1.71751658e+03 3.47764474e+04
 1.48788890e+08 2.04491499e+09 1.64060804e+09 1.72870951e+08]
[60. 60. 60. 60. 60. 60. 60. 47. 60. 60. 60. 60.]
[1.81000000e+02 2.42000000e+02 1.03532960e+04 7.11238475e+03
 3.12673300e+03 6.55379100e+03 8.37187500e+03 4.66385390e+03
 2.67026100e+06 2.79845000e+06 4.76408900e+06 1.38687800e+06]


#### Checking correlation using pearson method

In [13]:
corrMat = Statistics.corr(featuresRDD, method = 'pearson')
corrDf = pd.DataFrame(corrMat)
corrDf.index, corrDf.columns = colName, colName

corrDf.columns
corrDf.index
corrDf

,WkOfMth,DayOfWk,NonUrgentOrd,UrgentOrd,OrdTpA,OrdTpB,OrdTpC,FiscalSecOrd,OrdFrmTrafCtrl,BankOrd1,BankOrd2,BankOrd3
WkOfMth,1.000000,-0.207791,0.243472,0.107957,0.256115,0.312767,-0.041582,0.000941,-0.194088,0.392310,0.147086,-0.157059
DayOfWk,-0.207791,1.000000,-0.416331,-0.499372,-0.068894,-0.376512,-0.448823,-0.135058,-0.339485,-0.051815,-0.577035,-0.012251
NonUrgentOrd,0.243472,-0.416331,1.000000,0.558294,0.561397,0.827186,0.752627,-0.053814,0.246937,0.732357,0.788192,0.132857
UrgentOrd,0.107957,-0.499372,0.558294,1.000000,0.419749,0.508669,0.752069,-0.004690,0.255034,0.236350,0.651436,0.024818
OrdTpA,0.256115,-0.068894,0.561397,0.419749,1.000000,0.438734,0.218651,0.065084,-0.151762,0.675328,0.294374,0.230369
OrdTpB,0.312767,-0.376512,0.827186,0.508669,0.438734,1.000000,0.523598,-0.117874,0.127111,0.592845,0.713674,0.067325
OrdTpC,-0.041582,-0.448823,0.752627,0.752069,0.218651,0.523598,1.000000,0.010426,0.442404,0.330186,0.718739,0.031053
FiscalSecOrd,0.000941,-0.135058,-0.053814,-0.004690,0.065084,-0.117874,0.010426,1.000000,0.201770,0.003972,-0.051467,0.295435
OrdFrmTrafCtrl,-0.194088,-0.339485,0.246937,0.255034,-0.151762,0.127111,0.442404,0.201770,1.000000,-0.162309,0.240450,0.231614
BankOrd1,0.392310,-0.051815,0.732357,0.236350,0.675328,0.592845,0.330186,0.003972,-0.162309,1.000000,0.262905,0.221335


#### VectorAssembler

In [14]:
df1.show()

+-------+-------+------------+------------------+------+-------+-------+------------+--------------+--------+--------+--------+-------+
|WkOfMth|DayOfWk|NonUrgentOrd|         UrgentOrd|OrdTpA| OrdTpB| OrdTpC|FiscalSecOrd|OrdFrmTrafCtrl|BankOrd1|BankOrd2|BankOrd3| Target|
+-------+-------+------------+------------------+------+-------+-------+------------+--------------+--------+--------+--------+-------+
|      1|      4|     316.307|            223.27|61.543|175.586|302.448|         0.0|         65556|   44914|  188411|   14793|539.577|
|      1|      5|     128.633|            96.042|38.058| 56.037| 130.58|         0.0|         40419|   21399|   89461|    7679|224.675|
|      1|      6|      43.651|            84.375|21.826| 25.125| 82.461|       1.386|         11992|    3452|   21305|   14947|129.412|
|      2|      2|     171.297|           127.667|41.542|113.294|162.284|      18.156|         49971|   33703|   69054|   18423| 317.12|
|      2|      3|      90.532|           113.526

In [15]:
features = df1.drop('Target')
assembler = VectorAssembler(inputCols=features.columns, outputCol='features')
output = assembler.transform(df1)
output.select('features', 'Target').show(truncate = False)

+--------------------------------------------------------------------------------------------------+-------+
|features                                                                                          |Target |
+--------------------------------------------------------------------------------------------------+-------+
|[1.0,4.0,316.307,223.27,61.543,175.586,302.448,0.0,65556.0,44914.0,188411.0,14793.0]              |539.577|
|[1.0,5.0,128.633,96.042,38.058,56.037,130.58,0.0,40419.0,21399.0,89461.0,7679.0]                  |224.675|
|[1.0,6.0,43.651,84.375,21.826,25.125,82.461,1.386,11992.0,3452.0,21305.0,14947.0]                 |129.412|
|[2.0,2.0,171.297,127.667,41.542,113.294,162.284,18.156,49971.0,33703.0,69054.0,18423.0]           |317.12 |
|[2.0,3.0,90.532,113.526,37.679,56.618,116.22,6.459,48534.0,19646.0,16411.0,20257.0]               |210.517|
|[2.0,4.0,110.925,96.36,30.792,50.704,125.868,79.0,52042.0,8773.0,47522.0,24966.0]                 |207.364|
|[2.0,5.0,144.124,1

#### Standard Scaling

In [16]:
label = df1.select('Target')
label.show()

+-------+
| Target|
+-------+
|539.577|
|224.675|
|129.412|
| 317.12|
|210.517|
|207.364|
|263.043|
|248.958|
|344.291|
|248.428|
| 281.42|
|243.568|
|308.178|
|363.402|
|336.872|
|246.992|
| 308.88|
|233.126|
| 404.38|
| 298.56|
+-------+
only showing top 20 rows



In [17]:
features = df1.drop('Target')

In [18]:
colNames = features.columns
featuresRDD = features.rdd.map(lambda row: row[0:])

In [19]:
featuresRDD.collect()

[(1,
  4,
  316.307,
  223.27,
  61.543,
  175.586,
  302.448,
  0.0,
  65556,
  44914,
  188411,
  14793),
 (1,
  5,
  128.633,
  96.042,
  38.058,
  56.037,
  130.58,
  0.0,
  40419,
  21399,
  89461,
  7679),
 (1,
  6,
  43.651,
  84.375,
  21.826,
  25.125,
  82.461,
  1.386,
  11992,
  3452,
  21305,
  14947),
 (2,
  2,
  171.297,
  127.667,
  41.542,
  113.294,
  162.284,
  18.156,
  49971,
  33703,
  69054,
  18423),
 (2,
  3,
  90.532,
  113.526,
  37.679,
  56.618,
  116.22,
  6.459,
  48534,
  19646,
  16411,
  20257),
 (2,
  4,
  110.925,
  96.36,
  30.792,
  50.704,
  125.868,
  79.0,
  52042,
  8773,
  47522,
  24966),
 (2,
  5,
  144.124,
  118.919,
  43.304,
  66.371,
  153.368,
  0.0,
  46573,
  33597,
  48269,
  20973),
 (2,
  6,
  119.379,
  113.87,
  38.584,
  85.961,
  124.413,
  15.709,
  35033,
  26278,
  56665,
  18502),
 (3,
  2,
  218.856,
  124.381,
  33.973,
  148.274,
  162.044,
  1.054,
  66612,
  19461,
  103376,
  10458),
 (3,
  3,
  146.518,
  101.045,
 

In [20]:
scaler1 = StandardScaler().fit(featuresRDD)

In [21]:
scaledFeatures = scaler1.transform(featuresRDD)

In [22]:
for data in scaledFeatures.collect():
    print(data)

[0.7799691984347584,2.853526011061085,4.550800863796541,8.266170341711128,3.2683638968813797,3.4604098576424716,7.297939324655062,0.0,5.374365457059328,0.9932169106789801,4.6516157942976335,1.1251106774829749]
[0.7799691984347584,3.566907513826356,1.8506804070499245,3.5557823798925967,2.021146079773679,1.1043647397441207,3.1508388781326317,0.0,3.3136017665641737,0.47321210917797335,2.2086725327802545,0.5840414312439508]
[0.7799691984347584,4.280289016591627,0.6280196407464356,3.1238326805297456,1.1591133096100774,0.4951579150573912,1.989748236557627,0.007432255363002241,0.9831196314762258,0.07633666063285033,0.5259919776314073,1.1368234500329903]
[1.5599383968695169,1.4267630055305425,2.4644997915498426,4.72664114755782,2.206170856218356,2.2327729682989883,3.915842674979905,0.09735932782876529,4.096687050074923,0.7452996736120957,1.7048509750461955,1.4011974590190528]
[1.5599383968695169,2.1401445082958137,1.302510231519468,4.203096046101569,2.0010185280307025,1.1158149585957962,2.8043

#### PCA
Principal component analysis (PCA) is an unsupervised technique used to preprocess and reduce the dimensionality of high-dimensional datasets while preserving the original structure and relationships inherent to the original dataset so that machine learning models can still learn from them and be used to make accurate predictions.

In [23]:
pca = PCA(k=3)
pcaModel = pca.fit(scaledFeatures)

In [24]:
result = pcaModel.transform(scaledFeatures)
result.collect()

[DenseVector([-11.9308, 3.214, -1.0831]),
 DenseVector([-4.7769, 0.8855, -1.0311]),
 DenseVector([-1.9844, -0.4758, -1.1257]),
 DenseVector([-6.9123, 1.5959, -1.5182]),
 DenseVector([-4.5475, 1.1851, -1.9195]),
 DenseVector([-4.4935, 1.494, -2.2568]),
 DenseVector([-5.4321, 0.7319, -1.9554]),
 DenseVector([-4.7511, 0.0095, -1.5872]),
 DenseVector([-7.8088, 2.2655, -0.8568]),
 DenseVector([-5.821, 2.5669, -3.5891]),
 DenseVector([-6.3034, 0.7298, -1.6891]),
 DenseVector([-5.0552, 1.1181, -2.7261]),
 DenseVector([-5.9458, -0.6111, -2.0542]),
 DenseVector([-8.0683, 0.5722, -1.2993]),
 DenseVector([-7.8512, 0.9503, -1.9575]),
 DenseVector([-5.7347, 0.4201, -1.8801]),
 DenseVector([-6.7969, 0.7225, -1.0609]),
 DenseVector([-4.5329, -0.5677, -1.73]),
 DenseVector([-9.5997, 0.4598, 0.0062]),
 DenseVector([-7.3426, 2.0839, -7.6434]),
 DenseVector([-4.7593, 1.7836, -4.167]),
 DenseVector([-4.8194, 0.8424, -3.6057]),
 DenseVector([-5.6729, 0.6806, -2.3915]),
 DenseVector([-9.3816, 1.9671, -2.039

In [25]:
# store dense sector in a dataframe
df3 = result.map(lambda x: (x, )).toDF(['PCAFeatures'])
df3.show(truncate = False)

+-------------------------------------------------------------+
|PCAFeatures                                                  |
+-------------------------------------------------------------+
|[-11.930795189990496,3.2139857048807055,-1.0830945092899058] |
|[-4.776949597519387,0.8855231979118637,-1.0310693768692176]  |
|[-1.984433263216588,-0.47579913171671573,-1.1256657453203318]|
|[-6.9123235327165435,1.5958766350407576,-1.5182405909588446] |
|[-4.547512672421298,1.1850651274819135,-1.9194971040899553]  |
|[-4.493549228941707,1.4939638836413447,-2.256803483009176]   |
|[-5.432061751092074,0.7318508542421077,-1.9554449513172394]  |
|[-4.751076268442667,0.009547339019344644,-1.5872489533333836]|
|[-7.808778321197584,2.2654643722718952,-0.8568400955312063]  |
|[-5.820960208917008,2.5668563884632856,-3.5890758355419234]  |
|[-6.303400769367048,0.7298105632564738,-1.6890757439041073]  |
|[-5.055239456639968,1.1181053731672912,-2.7261221861922373]  |
|[-5.945761314195718,-0.6111307547660844